# Removing units without changing the output

A small classifier for machine-health readings has to run on a microcontroller, and every
parameter costs memory. Some of its hidden units are never used on the readings it will
see, and some duplicate the work of another unit. You will find the first kind with a
forward pass over the data, remove them, and then merge the second kind, checking each
time that the network's outputs do not change.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run these two cells first. The first holds the trained network's saved weights. The second
defines the lesson's `forward`, which returns the output and the cache with every `Z` and
`A`, and loads the network as `params` (layers of widths 8, 24, 16 and 4, `W` of shape
`(n_in, n_out)`) together with the 600 readings `X` it will be used on.

In [ ]:
# The classifier's saved weights, as its training script wrote them.
SAVED_WEIGHTS = """
{
 "W1": [
  [0.0882, 0.5993, -0.5696, 1.08495, 0.0, -0.2066, -0.0647, -0.7233, -0.0041, -0.0347, 1.0652, 0.3947, -0.01025, 0.445, -0.0132, 0.6277, -0.4344, 0.5321, 0.29965, -0.3762, 0.9713, -0.0264, 0.3677, -1.1465],
  [-0.3203, -0.8105, -0.3442, 1.43655, 0.0, 0.3676, -0.3276, -0.9577, -0.3633, 0.5719, 1.0918, 0.3563, -0.90825, -0.17, -0.0865, -0.3566, -0.4371, 0.2454, -0.40525, 0.4941, 0.4598, -0.173, -0.0773, 0.1149],
  [0.0013, 0.6993, -0.5292, 0.48975, 0.0, -1.1301, 0.6474, -0.3265, -0.2086, 0.2235, -0.5449, -0.9218, -0.5215, -0.2281, 0.6581, 0.3309, -0.2634, -0.9319, 0.34965, 0.7388, 0.5877, 1.3162, 0.2838, 0.1115],
  [-1.0513, 0.4357, -0.9073, 0.98565, 0.0, 0.1643, 0.3988, -0.6571, -0.0295, 0.7038, 0.2731, -0.1395, -0.07375, 0.7373, 0.3556, 0.4028, -0.1385, 0.358, 0.21785, -0.7058, 0.3676, 0.7112, -0.2633, 0.4859],
  [-0.156, -0.0881, -0.3077, 0.5829, 0.0, 0.7997, -0.1968, -0.3886, 0.5939, -0.9464, 1.246, 0.3956, 1.48475, -0.4625, 0.029, 0.0464, -0.1652, -0.2056, -0.04405, 0.1091, -1.3115, 0.058, 0.8073, -0.0149],
  [-0.0788, -0.0665, -0.395, 0.1542, 0.0, 0.229, -0.3171, -0.1028, 0.5317, 0.1539, -0.7631, 0.1833, 1.32925, -0.5202, -0.999, 0.4521, -0.8838, 0.3483, -0.03325, 1.5673, 0.2695, -1.998, -0.2335, 0.9866],
  [0.7748, 0.4969, -0.5388, -0.2958, 0.0, 0.1854, 0.1347, 0.1972, -1.2692, -0.6069, 0.0374, 0.1892, -3.173, -0.1824, 0.2427, -0.7734, -0.0651, -0.2335, 0.24845, 0.6334, -0.6986, 0.4854, 0.0606, -0.1215],
  [-1.3453, -1.4289, -0.2448, -0.7416, 0.0, 0.4776, -0.4279, 0.4944, 0.2074, -0.1622, 0.9122, 0.76, 0.5185, 0.4035, -1.0085, 0.5681, -0.499, -0.027, -0.71445, -0.4891, 0.6583, -2.017, -0.0499, -0.6704]
 ],
 "b1": [-0.1101, -0.3796, -0.05, -0.51795, 0.0, -0.094, -0.5027, 0.3453, 0.3223, 0.0298, 0.1807, -1.5753, 0.80575, 0.081, 0.4576, -0.0657, -0.2, -0.2329, -0.1898, -0.0501, -0.2005, 1.2152, 0.0344, 0.1289],
 "W2": [
  [-0.4913, -0.9897, 0.5832, -0.3716, 0.2663, 0.2515, 0.4194, -0.2221, -0.2861, -0.3994, -0.3299, -0.2356, -0.3442, -0.377, -0.2952, 0.3862],
  [0.0776, 1.9146, 0.0785, 0.334, 0.1261, 0.3991, 0.0874, -0.4102, -0.2981, 0.4029, 0.6382, -1.1871, -0.0525, 0.1297, -1.0661, -0.7281],
  [0.3454, 1.5996, 0.0986, 0.5508, -0.7353, -0.1815, 0.5616, -0.315, 1.0, -0.077, 0.5332, -0.1019, -0.0596, 0.129, 0.0662, -0.0124],
  [0.3256, -1.518, -0.3834, 0.2783, -0.4005, -0.4384, -0.4214, -0.8922, -0.318, -0.2947, -0.506, -0.1502, -0.6515, 0.2022, -0.0145, -0.0716],
  [0.4486, 1.188, 0.9313, 0.1798, 0.7403, 0.3876, -0.0606, -0.0429, 1.0, 0.1397, 0.396, 0.1384, 0.4027, -0.4851, -0.077, -0.4346],
  [0.0825, -1.0215, -0.3026, 0.235, 0.2013, 0.136, 0.1905, -0.9192, -0.5079, 0.5797, -0.3405, 0.1216, 0.6042, 0.0216, 0.0137, 0.8314],
  [-0.3192, -0.1326, 0.0561, -0.0062, -0.2688, 0.3931, 0.3761, -0.132, -0.4622, 0.2517, -0.0442, 0.5346, -0.9745, 0.1779, 0.0087, -0.0035],
  [0.4449, 0.7044, -0.2356, 0.1769, -0.7512, -0.0508, 0.0831, -0.1795, -0.8529, 0.6561, 0.2348, 0.3691, 0.3529, 0.6317, 0.0338, 0.1543],
  [0.1584, -0.8883, 0.6399, -0.2288, 0.7379, -0.2277, -0.3547, -0.0856, -0.1216, -0.0046, -0.2961, 0.2162, -0.1156, 0.7718, 0.1847, 0.0244],
  [0.2095, -0.3744, 0.061, -0.0727, 0.0434, 0.5434, -0.1267, -0.1695, -0.4192, -0.1618, -0.1248, 0.1494, -0.0985, -0.1126, 0.7206, 0.0873],
  [0.1795, 0.3435, -0.036, 0.0911, -0.0562, -0.5081, -0.3262, -0.1191, -0.309, -0.3753, 0.1145, -0.508, 0.2524, 0.1689, -0.0154, 0.3966],
  [-0.6139, -0.9012, 0.6393, 0.0518, -0.3141, 1.1449, 0.5547, -0.9358, -0.204, 0.0749, -0.3004, -0.0246, 0.599, 0.6152, -0.3386, 0.7816],
  [0.4995, 1.1166, 0.0819, -0.6348, 0.3382, -0.1702, 0.0587, -0.2628, -0.1729, -0.1958, 0.3722, -0.2228, -0.8809, -0.1519, 0.3445, -0.3092],
  [-0.0785, -2.3211, 0.0836, 0.4604, 0.6766, -0.3441, 0.6058, -0.127, -0.5357, 0.8141, -0.7737, 0.0278, 0.2786, 0.0955, 0.2728, -0.3697],
  [-0.1087, -1.6407, -0.1727, -0.1261, -0.635, -0.2422, 0.0776, -0.2265, -0.2288, 0.1362, -0.5469, 0.3164, 0.6102, -0.2412, -0.1507, -0.0476],
  [0.2296, 1.2195, -0.727, -0.3021, -0.1374, -0.2325, -0.2045, -0.3144, -0.0971, -0.9316, 0.4065, 0.3445, 0.0608, 0.3201, -0.0518, -0.1045],
  [0.3151, -2.1651, 1.0919, -0.3564, -0.3303, -0.381, -0.3634, -0.3103, 1.0, 0.0848, -0.7217, 0.1681, -0.4927, -0.1327, 0.7237, 0.4298],
  [0.2399, -0.3306, 0.6603, -0.117, 0.7616, -0.2232, -0.2414, -0.2196, -0.2339, -0.1116, -0.1102, -0.0522, -0.01, -0.2545, 0.012, 0.1671],
  [0.3093, -0.6861, -0.1035, 0.0799, -0.0203, -0.0213, -0.1788, -0.2118, -0.1035, 0.5991, -0.2287, 0.0012, -0.5223, -0.4203, -0.4718, -0.1443],
  [0.0697, -2.5932, 0.0944, 0.3282, 0.5293, 0.2016, 0.0523, -0.3219, -0.0604, 0.2895, -0.8644, -0.0845, 0.0179, -0.9522, -0.8556, -0.5936],
  [0.3569, 1.26, 0.0027, -0.5125, -0.2145, -0.0872, -0.1425, -0.5617, -0.0228, 0.8557, 0.42, 0.5589, 0.445, 0.0068, 0.1036, 0.0589],
  [-0.2435, -1.4772, 0.2179, -0.2006, -0.4535, 0.089, 0.1775, -0.2092, -0.5851, -0.0615, -0.4924, -0.0536, -0.5711, -0.3993, -0.6547, 0.0636],
  [-0.0551, 2.5791, -0.4289, -0.3995, -0.2359, 0.1035, -0.0809, -0.3439, -0.6052, -0.3201, 0.8597, 0.3662, 0.5274, -0.0245, 0.3775, 0.6108],
  [-0.5243, -0.3939, -0.0267, 0.1683, 0.1095, 0.146, 0.6633, -0.504, -0.1136, 0.1389, -0.1313, -0.2807, -0.328, -0.2542, -0.4215, 0.4868]
 ],
 "b2": [-0.154, 0.7221, -0.0698, 0.0892, 0.1776, 0.0488, -0.0172, -0.1, -0.05, 0.119, 0.2407, 0.0934, -0.1944, 0.4, 0.1574, 0.0728],
 "W3": [
  [-0.9675, 0.1603, 0.1831, -0.8417],
  [-0.0683, -0.1345, -0.1855, 0.2584],
  [0.4615, 0.065, -0.0293, -0.2735],
  [-0.7664, -0.666, -0.5167, 0.505],
  [0.1931, 0.1842, -0.0679, 0.0222],
  [0.3197, -0.4674, -0.4822, 0.4075],
  [0.5404, -0.6015, -0.6764, 0.1465],
  [0.0309, -0.462, 1.2218, 1.3334],
  [0.6932, -0.7457, -0.4318, -0.2719],
  [0.1291, 0.8672, -0.4364, -0.1394],
  [-0.1229, 0.6848, -0.4631, -0.0645],
  [0.3578, 0.4647, 0.0346, 0.1724],
  [-1.0139, 0.8389, 0.3538, 0.104],
  [0.9016, 0.1743, -0.8104, -0.3035],
  [-1.2665, 0.1537, 0.947, -0.2519],
  [-0.4426, 0.2284, -0.2571, -0.6195]
 ],
 "b3": [-0.1943, -0.0575, 0.0217, 0.0719]
}
"""

In [ ]:
import json

import numpy as np


def relu(Z):
    return np.where(Z > 0, Z, 0.0)


def softmax(Z):
    top = np.max(Z, axis=1)[:, None]               # each row's largest logit
    shifted = np.exp(Z - top)
    return shifted / shifted.sum(axis=1)[:, None]


def forward(X, params):
    """The lesson's forward pass: ReLU on every hidden layer, softmax on the last.
    params[l] = (W, b) with W of shape (n_in, n_out). Returns the output and the cache."""
    cache = {"A0": X}
    A = X
    for depth, (W, b) in enumerate(params, start=1):
        Z = A @ W + b
        A = softmax(Z) if depth == len(params) else relu(Z)
        cache[f"Z{depth}"] = Z
        cache[f"A{depth}"] = A
    return A, cache


def n_params(params):
    return int(sum(np.size(W) + np.size(b) for W, b in params))


def load_network():
    """The trained 8 -> 24 -> 16 -> 4 classifier, read from its saved weights."""
    saved = json.loads(SAVED_WEIGHTS)
    return [(np.array(saved[f"W{i}"]), np.array(saved[f"b{i}"])) for i in (1, 2, 3)]


params = load_network()
X = np.random.default_rng(80).random((600, 8))      # 600 readings, every feature scaled to [0, 1]
print("layer widths", [W.shape[1] for W, _ in params], "  parameters", n_params(params))

## Part 1 (coding task): Units the data never uses

Write `dead_units(params, X)`. It returns a list with one entry per hidden layer, in order:
a sorted integer array of the units in that layer whose activation is 0 on every row of
`X`. Use the forward pass and its cache.

In [ ]:
def dead_units(params, X):
    """One sorted index array per hidden layer: the units whose activation is 0 on every row of X."""
    # TODO
    raise NotImplementedError

In [ ]:
toy = [(np.array([[1.0, 0.0, -1.0], [-1.0, 0.0, -1.0]]), np.array([0.0, 0.0, -1.0])),
       (np.array([[1.0], [1.0], [1.0]]), np.array([0.0]))]
X_toy = np.array([[2.0, 1.0], [0.5, 1.5], [1.0, 1.0]])
d = dead_units(toy, X_toy)
assert len(d) == 1, f"expected one entry per hidden layer, got {len(d)}"
assert list(d[0]) != [0], "unit 0 of the small network is positive on the first row, so it is alive"
assert 1 in list(d[0]), "unit 1 of the small network has pre-activation exactly 0 on every row, so its activation is 0 and it is dead"
assert list(d[0]) == [1, 2], f"small network: expected units [1, 2], got {list(d[0])}"

d = dead_units(params, X)
assert [len(x) for x in d] != [], "expected one entry per hidden layer"
assert len(d) == 2, f"the network has two hidden layers, got {len(d)} entries"
got = [sorted(int(i) for i in x) for x in d]
assert 11 not in got[0], "unit 11 of layer 1 is active on 2 of the 600 rows, so it is not dead"
assert 2 not in got[1], "unit 2 of layer 2 is active on 1 row, so it is not dead"
assert got[0] == [2, 4, 16], f"layer 1: expected dead units [2, 4, 16], got {got[0]}"
assert got[1] == [7, 8], f"layer 2: expected dead units [7, 8], got {got[1]}"
print("dead units per hidden layer", got)
print("Part 1 passed")

## Part 2 (coding task): Remove them

Write `prune(params, dead)`. It returns a new parameter list in which the units listed in
`dead` (in the format of Part 1) are removed, with every array shrunk to match, and it
leaves `params` itself unchanged. On the readings `X`, the pruned network must give the
same outputs as the original.

In [ ]:
def prune(params, dead):
    """A new parameter list without the units in dead. params is left unchanged."""
    # TODO
    raise NotImplementedError

In [ ]:
before = [(W.copy(), b.copy()) for W, b in params]
dead = dead_units(params, X)
pruned = prune(params, dead)
assert all(np.array_equal(W, W0) and np.array_equal(b, b0) for (W, b), (W0, b0) in zip(params, before)), \
    "prune changed params itself: work on copies"
shapes = [W.shape for W, _ in pruned]
assert shapes == [(8, 21), (21, 14), (14, 4)], f"weight shapes {shapes}, expected [(8, 21), (21, 14), (14, 4)]"
assert [b.shape for _, b in pruned] == [(21,), (14,), (4,)], "each bias should lose the entries of the removed units"
out_full, _ = forward(X, params)
out_pruned, _ = forward(X, pruned)
assert np.allclose(out_pruned, out_full, atol=1e-12, rtol=0), \
    f"the pruned network changes the outputs on X by up to {np.abs(out_pruned - out_full).max():.3g}: check which rows and columns go"
print(f"parameters {n_params(params)} -> {n_params(pruned)}")
print("Part 2 passed")

## Part 3 (coding task): Merge units that do the same work

Write `merge_parallel(params, tol=1e-9)`. In each hidden layer, from the first to the last,
find every unit whose activation is a fixed number times the activation of an earlier unit
in the same layer, for every input to the network. Decide this from the units' incoming
weights and biases alone, up to a relative error `tol`. Merge each such unit into that
earlier unit: drop it, and change the earlier unit's outgoing weights so that the network's
output does not change for any input. Leave every other unit as it is, and return a new
parameter list.

You may assume no unit has all its incoming weights equal to zero. Run it on the pruned
network from Part 2.

In [ ]:
def merge_parallel(params, tol=1e-9):
    """Merge every hidden unit that can be folded into an earlier unit of its layer."""
    # TODO
    raise NotImplementedError

In [ ]:
rng = np.random.default_rng(3)
X_any = rng.normal(0, 3, (2000, 8))                  # any input at all, far outside [0, 1]

toy = [(np.array([[1.0, 2.0, -1.0, 0.5], [-2.0, -4.0, 2.0, -1.0]]), np.array([0.5, 1.0, -0.5, 0.25])),
       (np.array([[1.0, 0.0], [0.5, 1.0], [2.0, -1.0], [-1.0, 3.0]]), np.array([0.1, -0.2]))]
X_toy = rng.normal(0, 2, (500, 2))
m = merge_parallel(toy)
ref = forward(X_toy, toy)[0]
assert np.allclose(forward(X_toy, m)[0], ref, atol=1e-12), "on the small network the merged outputs differ for some inputs"
assert m[0][0].shape == (2, 2), (
    f"small network: units 1 and 3 are 2 and 0.5 times unit 0 and should be merged into it, "
    f"unit 2 should stay. Expected 2 units, got {m[0][0].shape[1]}")

merged = merge_parallel(pruned)
widths = [W.shape[1] for W, _ in merged]
out_p = forward(X_any, pruned)[0]
out_m = forward(X_any, merged)[0]
assert np.allclose(out_m, out_p, atol=1e-9, rtol=0), \
    f"merging changes the outputs by up to {np.abs(out_m - out_p).max():.3g} on some inputs"
assert widths[0] == 19, f"layer 1 should keep 19 units after merging, got {widths[0]}"
assert widths[1] == 13, f"layer 2 should keep 13 units after merging, got {widths[1]}"
assert np.allclose(forward(X, merged)[0], forward(X, params)[0], atol=1e-9), "on X the final network should match the original"
print(f"parameters {n_params(params)} -> {n_params(pruned)} after pruning -> {n_params(merged)} after merging")
print("Part 3 passed")

## Solution

### Part 1: Units the data never uses

In [ ]:
def dead_units(params, X):
    """One sorted index array per hidden layer: the units whose activation is 0 on every row of X."""
    _, cache = forward(X, params)
    return [np.flatnonzero((cache[f"A{l}"] == 0).all(axis=0)) for l in range(1, len(params))]

**Why it works.** A unit is dead on the data when its activation is 0 on every row, which
is one `all` down the rows of each cached `A`. Two of the dead units are easy to miss.
Unit 4 of layer 1 has pre-activation exactly 0, so a test of `Z < 0` keeps it, although
ReLU outputs 0 for it everywhere. Unit 8 of layer 2 has positive weights, but only from
units 2, 4 and 16 of layer 1, which are themselves dead, so it can only be found by running
the forward pass through layer 1 first, not by looking at its own weights. Unit 11 of
layer 1 and unit 2 of layer 2, active on two rows and on one row, are kept: removing them
would change those rows' outputs.

### Part 2: Remove them

In [ ]:
def prune(params, dead):
    """A new parameter list without the units in dead. params is left unchanged."""
    new = [(W.copy(), b.copy()) for W, b in params]
    for l, units in enumerate(dead):
        W, b = new[l]
        W_next, b_next = new[l + 1]
        new[l] = (np.delete(W, units, axis=1), np.delete(b, units))
        new[l + 1] = (np.delete(W_next, units, axis=0), b_next)
    return new

**Why it works.** In the `(n_in, n_out)` layout, unit `j` of layer `l` is column `j` of that
layer's `W` and entry `j` of its `b`, and its outgoing weights are row `j` of the next
layer's `W`. Deleting all three removes every trace of the unit. A dead unit's activation is
0 on every row of `X`, so its outgoing row only ever added zeros, and every output on `X`
stays exactly the same. `np.delete` with the whole index array removes all the units at
once, so no index shifts after an earlier deletion. The result is exact only on `X`: on
readings outside $[0, 1]$ some of these units would fire.

### Part 3: Merge units that do the same work

In [ ]:
def merge_parallel(params, tol=1e-9):
    """Merge every hidden unit that can be folded into an earlier unit of its layer."""
    new = [(W.copy(), b.copy()) for W, b in params]
    for l in range(len(new) - 1):
        W, b = new[l]
        W_next, b_next = new[l + 1]
        V = np.vstack([W, b])                       # column j: unit j's incoming weights and bias
        keep = []
        for k in range(V.shape[1]):
            for j in keep:
                c = V[:, k] @ V[:, j] / (V[:, j] @ V[:, j])
                if c > 0 and np.linalg.norm(V[:, k] - c * V[:, j]) <= tol * np.linalg.norm(V[:, k]):
                    W_next[j] += c * W_next[k]      # unit k's output is c times unit j's
                    break
            else:
                keep.append(k)
        new[l] = (W[:, keep], b[keep])
        new[l + 1] = (W_next[keep], b_next)
    return new

**Why it works.** If unit `k` has incoming vector $c$ times unit `j`'s with $c > 0$, then
$z_k = c\,z_j$ and $\mathrm{ReLU}(c\,z_j) = c\,\mathrm{ReLU}(z_j)$, so $a_k = c\,a_j$ for
every input. Unit `k` sends $a_k$ times its outgoing row to the next layer, which equals
$a_j$ times $c$ times that row, so adding $c$ times row `k` to row `j` and dropping `k`
gives the next layer the same input. With $c < 0$ this fails: $\mathrm{ReLU}(-z)$ is not
$-\mathrm{ReLU}(z)$, and the two units fire on opposite sides of the same line. In the
network as loaded, unit 3 of layer 1 is $-1.5$ times unit 7, and merging it would change the
outputs on many inputs. Unit 21 has twice the incoming weights of unit 14 but not twice its
bias, so its activation is not a multiple of unit 14's, and it has to stay too. Both checks
come from the whole vector of weights and bias.
The merge is exact for every input, unlike the pruning, which is why the check feeds it
inputs far outside $[0, 1]$. Working through the layers in order matters because merging
in layer 1 rewrites the rows of layer 2's weights that the search in layer 2 then reads.